# Module 3 — Vegetation Indices & Time Series at a Point

**Goal:** extract an actual NDVI/EVI time series for a single location and get it into a pandas DataFrame you can plot and analyze — the exact building block the glyphosate detector needs, just at 1-point scale before we scale to thousands of COMTRS polygons.

**Time:** ~1.5 hours


In [ ]:
import ee
try:
    ee.Initialize(project='YOUR-GCP-PROJECT-ID')
except Exception:
    ee.Authenticate()
    ee.Initialize(project='YOUR-GCP-PROJECT-ID')

import pandas as pd


### Reducers at a point — `.reduceRegion()`

A reducer collapses many pixel values down to one number (mean, median, max, ...). `.reduceRegion()` runs a reducer over the pixels of an image that fall inside a geometry. For a point geometry at native resolution, this effectively reads the value of the single pixel that contains it.

In [ ]:
point = ee.Geometry.Point(-121.5, 38.5)

img = ee.Image('COPERNICUS/S2_SR_HARMONIZED/20210715T184919_20210715T185724_T10SFG')
ndvi = img.normalizedDifference(['B8', 'B4']).rename('NDVI')

value = ndvi.reduceRegion(reducer=ee.Reducer.mean(), geometry=point, scale=10).getInfo()
print(value)


### Turning a whole collection into a time series

The pattern: `.map()` a function over the collection that computes NDVI **and** reduces it to a single number + date, producing a `FeatureCollection` of `{date, ndvi}` pairs with no geometry (or a point) attached. Then pull that down once with `.getInfo()` — one round trip instead of one per image.

In [ ]:
def add_ndvi_at_point(image, point):
    ndvi = image.normalizedDifference(['B8', 'B4'])
    mean_val = ndvi.reduceRegion(ee.Reducer.mean(), point, 10).get('nd')
    return ee.Feature(None, {
        'date': image.date().format('YYYY-MM-dd'),
        'ndvi': mean_val
    })

def mask_s2_clouds(image):
    qa = image.select('QA60')
    mask = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))
    return image.updateMask(mask)

collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
              .filterDate('2021-01-01', '2021-12-31')
              .filterBounds(point)
              .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 50))
              .map(mask_s2_clouds))

fc = collection.map(lambda img: add_ndvi_at_point(img, point))

# one network round-trip for the whole year
records = fc.getInfo()['features']
df = pd.DataFrame([r['properties'] for r in records]).dropna()
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date')
df.head()


In [ ]:
df.plot(x='date', y='ndvi', marker='o', linestyle='-', figsize=(10,4), title='NDVI time series')


### Exercise 3.1
The chart above will have gaps and outlier dips from residual clouds. Add a `ndvi > 0` and `ndvi < 1` sanity filter, and also filter out points where the pixel had fewer than some minimum number of unmasked observations (hint: use `ee.Reducer.count()` alongside `.mean()` via `.combine()`, or just check `mean_val` isn't null which you're already doing). Re-plot.


In [ ]:
# TODO: your code here


### Exercise 3.2
Repeat the extraction for **EVI** instead of NDVI (formula: `2.5 * (NIR - RED) / (NIR + 6*RED - 7.5*BLUE + 1)`, bands B8/B4/B2 for Sentinel-2, all reflectance-scaled to 0-1 — remember to `.divide(10000)` first). Plot NDVI and EVI on the same axes for comparison.


In [ ]:
# TODO: your code here


### Project 3 — "Point time-series tool"

Write a reusable function:

```python
def get_index_timeseries(lon, lat, start, end, index='NDVI', collection_id='COPERNICUS/S2_SR_HARMONIZED', max_cloud_pct=50) -> pd.DataFrame
```

that returns a clean DataFrame with columns `date`, `<index>` for any point/date range/index (support at least NDVI and EVI). This function — almost verbatim — is what the glyphosate pipeline uses per-COMTRS-chunk once you scale up in Module 4/5. Test it on 2-3 different points (e.g. a known agricultural field vs. a forest) and sanity-check the seasonal shape looks like vegetation phenology (green-up in spring, peak in summer, senescence in fall) rather than noise.


In [ ]:
def get_index_timeseries(lon, lat, start, end, index='NDVI',
                          collection_id='COPERNICUS/S2_SR_HARMONIZED', max_cloud_pct=50):
    # TODO
    pass


---
**Checkpoint:** you should be able to go from "a lon/lat and a date range" to "a pandas DataFrame of vegetation index over time" without looking anything up. That's the atomic operation of the whole glyphosate pipeline — next module is about doing this for thousands of polygons at once instead of one point at a time, which needs a different, much more efficient approach (`reduceRegions`, not a Python loop over `reduceRegion`).